# E1: Which segmentation model?

**Question.** Which model gives the best alpha for Modelia-like inputs (people with hair, fashion products), at what speed and memory?

**Data** (`data/e1_subset.md`): 45 images with ground truth in 3 groups + 10 real fashion photos without ground truth.

| Group | n | Ground truth | Judged by |
|---|---:|---|---|
| A_people (PPM-100) | 15 | soft alpha matte | IoU, Boundary IoU, MAE |
| B_products (DIS-VD Accessories) | 20 | binary mask | IoU, Boundary IoU, MAE |
| C_structure (DIS-VD, hand-picked) | 10 | binary mask | same, reported apart (stress set) |
| D_real (fashion photos) | 10 | none | visual only: every result is shown and saved |

**Models**: BiRefNet, BiRefNet_lite, BiRefNet-matting, BEN2, InSPyReNet, IS-Net, U²-Net (all run here), RMBG-2.0 (public Space, reference only: non-commercial licence).

**Hypotheses (written before the run)**
1. BiRefNet (general) has the best IoU / Boundary IoU on B and C (it is trained on DIS5K-TR).
2. On A (hair), the matting-style models (BiRefNet-matting, BEN2) have the lowest MAE, but BiRefNet-matting's number is optimistic: PPM-100 is in its training set.
3. BiRefNet_lite is clearly faster than BiRefNet for a small accuracy loss.
4. U²-Net (320 px input) is clearly the worst on Boundary IoU.
5. Every model struggles on D09 (white top on white background) and on the bottom of the black dress in D10 (low light).

**How the notebook works.** The GPU part (section 3) runs each model once and saves its alpha as a PNG. Everything after it (metrics, tables, figures) only reads those PNGs, so it can be re-run anywhere, also locally on CPU, without the models.

| Where | Run |
|---|---|
| Colab GPU (VS Code Colab extension) | sections 1 to 8 |
| Local kernel (`.venv`) | sections 1, 2, then 5 to 7 after fetching the results (`python experiments/fetch_e1_results.py`); section 9 (CPU timing) |

## 1. Setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MedSahbiBenKalia/modelia-bg-change"

if IN_COLAB:
    ROOT = Path("/content/modelia-bg-change")
    if ROOT.exists():
        !git -C {ROOT} pull -q
    else:
        !git clone -q {REPO_URL} {ROOT}
    # Lower bounds only in requirements.txt, so Colab keeps its CUDA torch.
    # rembg[gpu] brings onnxruntime-gpu; BEN2 is not on PyPI.
    !pip install -q -r {ROOT}/requirements.txt "rembg[gpu]" transparent-background gradio_client git+https://github.com/PramaLLC/BEN2.git
else:
    # Repo root = first parent folder that holds requirements.txt (works from experiments/ or the root).
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))
print("repo:", ROOT, "| colab:", IN_COLAB)

In [ ]:
import gc
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

from bgchange.composite import over_colour
from bgchange.io import load_alpha, load_rgb, parse_hex, save_alpha
from bgchange.metrics import boundary_iou, iou, mae

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32  # fp16 is slow / unsupported on CPU
GPU_NAME = torch.cuda.get_device_name() if DEVICE == "cuda" else "none"
print("device:", DEVICE, DTYPE, "| gpu:", GPU_NAME, "| torch", torch.__version__)

### Data and results folders

On Colab the images come from Google Drive (`MyDrive/modelia-e1`, uploaded from `data/e1/`) and results are written next to them in `modelia-e1/results/`, so they survive a disconnect. If mounting Drive fails, the shared folder is downloaded with `gdown` instead, and results stay on the Colab disk only (download them before closing the session).

Locally: `data/e1/` and `results/e1/`.

In [ ]:
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1XorV6sxMkklR9busqG-He3I4Ay0qsZAO"


def find_e1_root(base: Path) -> Path:
    # The folder that holds A_people/, B_products/, ... (the upload may have added one level).
    for cand in [base, *base.glob("*"), *base.glob("*/*")]:
        if (cand / "A_people").is_dir():
            return cand
    raise FileNotFoundError(f"no A_people/ folder under {base}")


if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        mydrive = Path("/content/drive/MyDrive")
        base = next(p for p in [mydrive / "modelia-e1", *mydrive.glob("*/modelia-e1"), *mydrive.glob("*/*/modelia-e1")] if p.is_dir())
        RESULTS = base / "results"
    except Exception as e:  # mount not available, or folder not found
        print("Drive not used:", repr(e), "-> downloading the shared folder with gdown")
        import gdown
        base = Path("/content/modelia-e1")
        if not (base.exists() and any(base.iterdir())):
            gdown.download_folder(DRIVE_FOLDER_URL, output=str(base), quiet=True)
        RESULTS = Path("/content/e1_results")
        print("WARNING: results are only on the Colab disk:", RESULTS)
    E1_ROOT = find_e1_root(base)
else:
    E1_ROOT = ROOT / "data" / "e1"
    RESULTS = ROOT / "results" / "e1"

ALPHAS = RESULTS / "alphas"      # alphas/<model>/<image name>.png
FIGURES = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
print("images :", E1_ROOT)
print("results:", RESULTS)

## 2. The subset

In [ ]:
subset = pd.read_csv(ROOT / "data" / "e1_subset.csv", keep_default_na=False)


def local(rel: str) -> Path:
    # CSV paths are 'e1/<group>/...' relative to data/.
    return E1_ROOT / Path(rel).relative_to("e1")


missing = [r for r in subset["image"] if not local(r).exists()]
missing += [g for g in subset["gt"] if g and not local(g).exists()]
assert not missing, f"{len(missing)} files missing (upload not finished?), e.g. {missing[:3]}"

labelled = subset[subset["gt_type"] != "none"]
print(subset.groupby(["group", "gt_type"]).size().to_string())

## 3. Models

Every model is wrapped the same way: `load()` returns `predict(img) -> alpha`, a float array in [0, 1] **at the input's full size**, plus a short description of the backend. Pre- and post-processing (resize, normalisation, upsampling back) are inside `predict`, because the pipeline pays for them too.

| Model | Code | Input size | Licence |
|---|---|---|---|
| BiRefNet / _lite / -matting | `transformers`, `trust_remote_code` | 1024×1024 | MIT |
| BEN2 (base) | `ben2` | 1024×1024 | MIT |
| InSPyReNet | `transparent-background` (mode `base`) | 1024×1024 | MIT |
| IS-Net (general use) | `rembg` (ONNX) | 1024×1024 | Apache-2.0 |
| U²-Net | `rembg` (ONNX) | 320×320 | Apache-2.0 |

In [ ]:
from torchvision import transforms
from transformers import AutoModelForImageSegmentation

to_tensor_1024 = transforms.Compose([
    transforms.Resize((1024, 1024)),  # BiRefNet's training resolution
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNet stats
])


def load_birefnet(repo_id):
    torch.set_float32_matmul_precision("high")
    model = AutoModelForImageSegmentation.from_pretrained(repo_id, trust_remote_code=True)
    model = model.to(DEVICE, DTYPE).eval()  # cast explicitly: the hub weights are stored in fp16

    def predict(img):
        x = to_tensor_1024(img).unsqueeze(0).to(DEVICE, DTYPE)
        with torch.no_grad():
            logits = model(x)[-1]  # list of side outputs, the last one is the final prediction
            alpha = F.interpolate(logits.sigmoid(), size=img.size[::-1], mode="bilinear", align_corners=False)
        return alpha[0, 0].float().cpu().numpy()

    return predict, f"torch {DEVICE} {str(DTYPE).replace('torch.', '')}"


def load_ben2():
    from ben2 import BEN_Base
    model = BEN_Base.from_pretrained("PramaLLC/BEN2").to(DEVICE).eval()

    def predict(img):
        fg = model.inference(img.copy(), refine_foreground=False)  # copy: BEN2 calls putalpha() on its input
        return np.asarray(fg.getchannel("A"), np.float32) / 255

    return predict, f"torch {DEVICE} (fp16 autocast on cuda)"


def load_inspyrenet():
    from transparent_background import Remover
    remover = Remover(mode="base", device="cuda:0" if DEVICE == "cuda" else "cpu")

    def predict(img):
        sal = remover.process(img, type="map")  # saliency map, no threshold -> soft
        return np.asarray(sal.convert("L"), np.float32) / 255

    return predict, f"torch {DEVICE}"


def load_rembg(model_name):
    import onnxruntime as ort
    from rembg import new_session, remove
    if DEVICE == "cuda" and hasattr(ort, "preload_dlls"):
        ort.preload_dlls()  # use the CUDA / cuDNN libraries that come with torch
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if DEVICE == "cuda" else ["CPUExecutionProvider"]
    session = new_session(model_name, providers=providers)
    used = session.inner_session.get_providers()[0]  # what ONNX Runtime really uses (it can fall back to CPU)

    def predict(img):
        mask = remove(img, session=session, only_mask=True)  # uint8 PIL image, soft (no post-processing)
        return np.asarray(mask, np.float32) / 255

    return predict, f"onnxruntime {used}"


MODELS = {
    "BiRefNet": lambda: load_birefnet("ZhengPeng7/BiRefNet"),
    "BiRefNet_lite": lambda: load_birefnet("ZhengPeng7/BiRefNet_lite"),
    "BiRefNet-matting": lambda: load_birefnet("ZhengPeng7/BiRefNet-matting"),
    "BEN2": load_ben2,
    "InSPyReNet": load_inspyrenet,
    "IS-Net": lambda: load_rembg("isnet-general-use"),
    "U2-Net": lambda: load_rembg("u2net"),
}
INPUT_SIZE = {"U2-Net": 320}  # all others: 1024

### Run every model on all 55 images

One model at a time: load, one warm-up image (not timed: CUDA kernels and cuDNN autotuning), then every image is timed from PIL image in to full-size alpha out (image decoding excluded). The alpha is saved to `alphas/<model>/<name>.png`.

- **Peak GPU memory** = `torch.cuda.max_memory_allocated` from before loading to the end (weights + activations). ONNX Runtime (IS-Net, U²-Net) allocates outside torch, so it is not measured: `NaN`, not 0.
- **Resume**: a model whose alphas and timing file already exist is skipped (Colab can disconnect). Delete `results/timing/<model>.csv` to force a re-run.
- On CPU this section is skipped unless `RUN_ON_CPU = True` (7 models × 55 images on a laptop CPU takes hours).

In [ ]:
RUN_ON_CPU = False
TIMING = RESULTS / "timing"


def run_model(name):
    out_dir, timing_csv = ALPHAS / name, TIMING / f"{name}.csv"
    if timing_csv.exists() and all((out_dir / f"{n}.png").exists() for n in subset["name"]):
        print(f"{name}: already done, skipped")
        return
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    predict, backend = MODELS[name]()
    load_s = time.perf_counter() - t0
    predict(load_rgb(local(subset["image"].iloc[0])))  # warm-up, not timed

    rows = []
    for r in subset.to_dict("records"):
        img = load_rgb(local(r["image"]))
        t = time.perf_counter()
        alpha = predict(img)
        rows.append({"model": name, "name": r["name"], "group": r["group"], "width": img.width,
                     "height": img.height, "seconds": time.perf_counter() - t})
        save_alpha(alpha, out_dir / f"{r['name']}.png")

    peak = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" and backend.startswith("torch") else np.nan
    TIMING.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows).assign(backend=backend, gpu=GPU_NAME, load_seconds=load_s, peak_gpu_mb=peak)
    df.to_csv(timing_csv, index=False)
    print(f"{name}: {backend} | median {df.seconds.median():.3f} s/img | peak GPU {peak:.0f} MB")
    del predict
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


if DEVICE == "cuda" or RUN_ON_CPU:
    for name in MODELS:
        run_model(name)
else:
    print("No GPU: prediction skipped. Run this section on Colab, or set RUN_ON_CPU = True.")

### RMBG-2.0 (reference only)

Non-commercial licence, so not a candidate for the pipeline; called through the public Hugging Face Space (no token). The free ZeroGPU quota is small, so by default only group D is sent. Times include upload and queueing: they are **not** comparable with the table above and are not reported. Errors (quota) are printed and skipped.

In [ ]:
import tempfile

RMBG_GROUPS = ["D_real"]  # add "A_people", "B_products", ... if the quota allows


def rmbg_alpha(client, img):
    with tempfile.TemporaryDirectory() as tmp:
        f = Path(tmp) / "in.jpg"
        img.save(f, quality=95)
        _, png_path = client.predict(image=handle_file(str(f)), api_name="/image")  # (slider, transparent PNG)
        alpha = Image.open(png_path).convert("RGBA").getchannel("A").resize(img.size, Image.BILINEAR)
    return np.asarray(alpha, np.float32) / 255


if IN_COLAB or RUN_ON_CPU:
    from gradio_client import Client, handle_file
    client = Client("briaai/BRIA-RMBG-2.0", httpx_kwargs={"timeout": 120})
    for r in subset[subset["group"].isin(RMBG_GROUPS)].to_dict("records"):
        out = ALPHAS / "RMBG-2.0_API" / f"{r['name']}.png"
        if out.exists():
            continue
        try:
            save_alpha(rmbg_alpha(client, load_rgb(local(r["image"]))), out)
            print("ok", r["name"])
        except Exception as e:
            print("failed", r["name"], repr(e)[:200])

## 4. Speed and memory (GPU)

Median seconds per image (the mean is pulled up by the few 5000+ px images). `load s` = loading the weights (download excluded on a re-run).

In [ ]:
def md_table(df: pd.DataFrame, floatfmt="{:.3f}") -> str:
    # Markdown table without the tabulate dependency.
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        cells = [floatfmt.format(v) if isinstance(v, float) else str(v) for v in row]
        lines.append("| " + " | ".join([str(idx), *cells]) + " |")
    return "\n".join(lines)


timing_files = sorted(TIMING.glob("*.csv")) if TIMING.exists() else []
if timing_files:
    timing = pd.concat(map(pd.read_csv, timing_files))
    speed = timing.groupby("model").agg(
        backend=("backend", "first"), gpu=("gpu", "first"),
        median_s=("seconds", "median"), mean_s=("seconds", "mean"), max_s=("seconds", "max"),
        load_s=("load_seconds", "first"), peak_gpu_mb=("peak_gpu_mb", "first"))
    speed.insert(2, "input_px", [INPUT_SIZE.get(m, 1024) for m in speed.index])
    speed = speed.loc[[m for m in MODELS if m in speed.index]]
    speed.to_csv(RESULTS / "e1_speed_gpu.csv")
    (RESULTS / "e1_speed_gpu.md").write_text(md_table(speed), encoding="utf-8")
    display(speed.round(3))
else:
    print("no timing files yet")

## 5. Accuracy on the labelled groups (A, B, C)

Per image and model: **IoU** (alpha and ground truth thresholded at 0.5), **Boundary IoU** (IoU on a band of 2 % of the image diagonal inside each outline; Cheng et al., CVPR 2021) and **MAE** on the soft alpha. The ground truth is compared at its full resolution.

Training-data overlap (checked on the model cards, see `data/e1_subset.md`) is marked with `*` in the summary.

In [ ]:
rows = []
model_dirs = sorted(d for d in ALPHAS.iterdir() if d.is_dir()) if ALPHAS.exists() else []
for r in labelled.to_dict("records"):
    gt = load_alpha(local(r["gt"]))
    for d in model_dirs:
        p = d / f"{r['name']}.png"
        if p.exists():
            pred = load_alpha(p)
            rows.append({"model": d.name, "group": r["group"], "name": r["name"],
                         "iou": iou(pred, gt), "biou": boundary_iou(pred, gt), "mae": mae(pred, gt)})
metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS / "e1_metrics.csv", index=False)
print(len(metrics), "rows ->", RESULTS / "e1_metrics.csv")

In [ ]:
LEAKS = {  # (model, group): reason; see data/e1_subset.md
    ("BiRefNet-matting", "A_people"): "PPM-100 is in its training set",
    ("BiRefNet", "B_products"): "DIS-VD is its validation set", ("BiRefNet", "C_structure"): "DIS-VD is its validation set",
    ("BiRefNet_lite", "B_products"): "DIS-VD is its validation set", ("BiRefNet_lite", "C_structure"): "DIS-VD is its validation set",
    ("BEN2", "B_products"): "trained on DIS5K, splits not stated", ("BEN2", "C_structure"): "trained on DIS5K, splits not stated",
    ("RMBG-2.0_API", "B_products"): "may be trained on DIS-VD (claim by the BEN2 authors)",
    ("RMBG-2.0_API", "C_structure"): "may be trained on DIS-VD (claim by the BEN2 authors)",
}
ORDER = [m for m in [*MODELS, "RMBG-2.0_API"] if m in set(metrics.get("model", []))]

if len(metrics):
    # Mean per model and group, plus "A+B" = the fashion-relevant images pooled (35 images).
    per_group = metrics.groupby(["model", "group"])[["iou", "biou", "mae"]].mean()
    ab = metrics[metrics.group.isin(["A_people", "B_products"])].groupby("model")[["iou", "biou", "mae"]].mean()
    ab["group"] = "A+B"
    summary = pd.concat([per_group.reset_index(), ab.reset_index()])
    summary.to_csv(RESULTS / "e1_summary.csv", index=False)

    lines = []
    for metric, better in [("iou", "higher"), ("biou", "higher"), ("mae", "lower")]:
        t = summary.pivot(index="model", columns="group", values=metric).loc[ORDER]
        t = t[[g for g in ["A_people", "B_products", "C_structure", "A+B"] if g in t.columns]]
        shown = t.map(lambda v: f"{v:.3f}").astype(object)
        for (m, g) in LEAKS:
            if m in shown.index and g in shown.columns:
                shown.loc[m, g] += " *"
        lines += [f"### {metric.upper()} ({better} is better)", "", md_table(shown), ""]
        print(metric.upper(), f"({better} is better)")
        display(shown)
    lines += ["`*` training-data overlap:", ""] + [f"- {m} on {g}: {why}" for (m, g), why in LEAKS.items()]
    (RESULTS / "e1_summary.md").write_text("\n".join(lines), encoding="utf-8")

## 6. Labelled hard cases (picked by rule, not by eye)

For each group: the image with the **lowest mean IoU** over the 7 local models, and the image where the models **disagree most** (highest std of IoU). Each row: input, ground truth, then each model's alpha. A second figure zooms on the window where the mean error is largest (useful for hair).

In [ ]:
import cv2

SHOW = [m for m in MODELS if m in set(metrics.get("model", []))]


def thumb(a, size=512):
    im = a if isinstance(a, Image.Image) else Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8))
    im = im.copy()
    im.thumbnail((size, size))
    return im


def pick_hard_cases(metrics):
    local_m = metrics[metrics.model.isin(MODELS)]
    stats = local_m.groupby(["group", "name"]).iou.agg(["mean", "std"]).fillna(0).reset_index()  # std is NaN with one model
    picks = []
    for g, s in stats.groupby("group"):
        hardest = s.loc[s["mean"].idxmin(), "name"]
        rest = s[s["name"] != hardest]
        picks += [(g, hardest, "lowest mean IoU"), (g, rest.loc[rest["std"].idxmax(), "name"], "most disagreement")]
    return picks


def worst_window(name, frac=0.25):
    # Top-left corner + size of the square window where the mean |alpha - gt| over the models is largest.
    r = labelled.set_index("name").loc[name]
    gt = load_alpha(local(r["gt"]))
    err = np.mean([np.abs(load_alpha(ALPHAS / m / f"{name}.png") - gt) for m in SHOW], axis=0)
    k = max(8, int(frac * min(gt.shape)))
    smooth = cv2.blur(err, (k, k))  # mean error of the k x k window centred on each pixel
    cy, cx = np.unravel_index(smooth.argmax(), smooth.shape)
    y0, x0 = np.clip(cy - k // 2, 0, gt.shape[0] - k), np.clip(cx - k // 2, 0, gt.shape[1] - k)
    return y0, x0, k


def grid(picks, zoom=False, path=None):
    cols = ["input", "ground truth", *SHOW]
    fig, axes = plt.subplots(len(picks), len(cols), figsize=(2.2 * len(cols), 2.4 * len(picks)), squeeze=False)
    for i, (g, name, why) in enumerate(picks):
        r = labelled.set_index("name").loc[name]
        panels = [load_rgb(local(r["image"])), load_alpha(local(r["gt"]))]
        panels += [load_alpha(ALPHAS / m / f"{name}.png") for m in SHOW]
        if zoom:
            y0, x0, k = worst_window(name)
            panels = [np.asarray(p)[y0:y0 + k, x0:x0 + k] for p in panels]
            panels[0] = Image.fromarray(panels[0])
        for j, (ax, p) in enumerate(zip(axes[i], panels)):
            ax.imshow(thumb(p), cmap=None if j == 0 else "gray", vmin=0, vmax=255)
            ax.set_xticks([]), ax.set_yticks([])
            if i == 0:
                ax.set_title(cols[j], fontsize=9)
            if j >= 2:
                m = metrics[(metrics.model == SHOW[j - 2]) & (metrics.name == name)].iloc[0]
                ax.set_xlabel(f"IoU {m.iou:.2f}  BIoU {m.biou:.2f}", fontsize=7)
        axes[i][0].set_ylabel(f"{g}\n{why}", fontsize=8)
    plt.tight_layout()
    if path:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=110)
    plt.show()


if len(metrics):
    picks = pick_hard_cases(metrics)
    pd.DataFrame(picks, columns=["group", "name", "why"]).to_csv(RESULTS / "e1_hard_cases.csv", index=False)
    grid(picks, path=FIGURES / "e1_labelled_hard_cases.png")
    grid(picks, zoom=True, path=FIGURES / "e1_labelled_hard_cases_zoom.png")

## 7. Real fashion photos (group D): every result

No ground truth, so these are judged by eye. Each model's alpha is composited on `BG_HEX` (a saturated colour makes halos and missing hair easy to see). One figure per photo; every composite is also saved to `composites/<model>/<name>.jpg`.

D09 and D10 have no known licence: their figures stay out of git (see `.gitignore`).

In [ ]:
BG_HEX = "#1E90FF"
bg_rgb = parse_hex(BG_HEX)
D_MODELS = [m for m in [*MODELS, "RMBG-2.0_API"] if (ALPHAS / m).is_dir()]

for r in subset[subset["group"] == "D_real"].to_dict("records"):
    img = load_rgb(local(r["image"]))
    panels, titles = [img], ["input"]
    for m in D_MODELS:
        p = ALPHAS / m / f"{r['name']}.png"
        if not p.exists():
            continue
        comp = over_colour(img, load_alpha(p), bg_rgb)
        out = RESULTS / "composites" / m / f"{r['name']}.jpg"
        out.parent.mkdir(parents=True, exist_ok=True)
        thumb(comp, 1600).save(out, quality=92)
        panels.append(comp), titles.append(m)
    if len(panels) == 1:
        continue
    fig, axes = plt.subplots(1, len(panels), figsize=(2.6 * len(panels), 3.6))
    for ax, p, t in zip(axes, panels, titles):
        ax.imshow(thumb(p, 800)), ax.set_title(t, fontsize=9), ax.axis("off")
    fig.suptitle(f"{r['name']}: {r['note']}", fontsize=9)
    plt.tight_layout()
    (FIGURES / "D").mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / "D" / f"{r['name']}.png", dpi=110)
    plt.show()

## 8. Pack the results (Colab)

Google Drive folders with more than 50 files cannot be listed by `gdown`, so the per-image folders are also zipped. Then, locally: `python experiments/fetch_e1_results.py` copies everything into `results/e1/`.

In [ ]:
import shutil

if IN_COLAB:
    for folder in ["alphas", "composites"]:
        if (RESULTS / folder).exists():
            shutil.make_archive(str(RESULTS / folder), "zip", RESULTS / folder)
    print(sorted(p.name for p in RESULTS.iterdir()))

## 9. CPU timing (local kernel only)

The Colab CPU is not the machine the CLI will run on, so CPU speed is measured locally on 5 images (random, seed 0). Peak memory on CPU = largest increase of the process RSS after an image, relative to before loading the model: a lower bound of the true peak (short spikes inside a forward pass are missed). Takes about 15 to 25 minutes.

In [ ]:
import psutil

RUN_CPU_TIMING = not IN_COLAB and DEVICE == "cpu"

if RUN_CPU_TIMING:
    proc, rows = psutil.Process(), []
    sample = subset.sample(5, random_state=0)
    for name, load in MODELS.items():
        gc.collect()
        rss0 = proc.memory_info().rss
        predict, backend = load()
        predict(load_rgb(local(sample["image"].iloc[0])))  # warm-up
        for r in sample.to_dict("records"):
            img = load_rgb(local(r["image"]))
            t = time.perf_counter()
            predict(img)
            rows.append({"model": name, "name": r["name"], "width": img.width, "height": img.height,
                         "seconds": time.perf_counter() - t, "rss_increase_mb": (proc.memory_info().rss - rss0) / 2**20})
        print(name, "done")
        del predict
    cpu = pd.DataFrame(rows)
    cpu.to_csv(RESULTS / "e1_timing_cpu.csv", index=False)
    speed_cpu = cpu.groupby("model").agg(median_s=("seconds", "median"), max_rss_increase_mb=("rss_increase_mb", "max")).loc[list(MODELS)]
    speed_cpu.to_csv(RESULTS / "e1_speed_cpu.csv")
    (RESULTS / "e1_speed_cpu.md").write_text(md_table(speed_cpu), encoding="utf-8")
    display(speed_cpu.round(2))

## 10. Result and decision

*To be written after the run, from the tables above: which hypotheses held, which did not, the chosen model and why (accuracy on A+B, speed, licence), and what still fails (group D).*